# 🕵️ The Case of the Vanishing Repayments
### Learn EDA in pandas by thinking like a detective

**The story.** You've just joined *Meridian Lending* as a junior analyst. The Head of Credit Risk drops a case file on your desk:

> *"About 1 in 10 of our personal loans defaults. Nobody knows why. The data is messy, someone has been sloppy with it, and I need answers by Friday. Find me the clues."*

Your job is **Exploratory Data Analysis (EDA)**: before you model or conclude anything, you *interrogate the data*. A good EDA is exactly like detective work:

| Detective move | Data move | pandas tools you'll learn |
|---|---|---|
| Arrive at the scene, look around | Shape, types, first rows | `shape`, `head`, `info`, `describe` |
| Notice what's missing | Missing values | `isna`, `sum`, `mean` |
| Spot liars and impossible alibis | Errors, outliers, duplicates | `duplicated`, `query`, `sort_values` |
| Preserve the evidence | Clean on a copy | `copy`, `str.strip`, `fillna`, `drop_duplicates` |
| Interview one witness at a time | Single-variable analysis | `value_counts`, `hist` |
| Connect the dots | Group & compare | `groupby`, `agg`, `pd.cut`, `crosstab` |
| Beware of red herrings | Correlation vs. causation | `corr`, careful thinking |
| Write the case report | Findings + next steps | your brain 🧠 |

## 🧠 How to use this notebook (this is the "10x" part)

Reading code feels like learning, but it mostly isn't. These habits are what actually make things stick:

1. **Predict before you run.** Every chapter has a 🔮 *Prediction* box. Write your guess down *before* running the cell. Being wrong is the most valuable moment, because that's when your brain updates.
2. **Type it, don't paste it.** For the 🎯 *Your turn* cells, struggle for a few minutes *before* opening the 🔓 solution.
3. **Explain in your own words.** After each chapter, answer the 🗣️ *Explain it* prompt in plain English, as if to a colleague.
4. **Tomorrow, retrieve.** Next day, try the ⚡ *Recall* questions **without looking**. Spaced retrieval is one of the most evidence-backed learning techniques there is.
5. **Always ask "so what?"** A number is not an insight until you can say what it means and what you'd do about it.

> ⚠️ **Don't read the cell below.** It builds the crime scene. Reading it would spoil the mystery. Just run it.

## 🎬 Chapter 0: Set up the crime scene

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.rcParams['figure.figsize'] = (7, 4)

In [ ]:
# 🚫 SPOILER ALERT: don't read this cell. Just run it.
def make_case_file(n=3000, seed=7):
    rng = np.random.default_rng(seed)
    age = np.clip(rng.normal(41, 12, n), 19, 75).round().astype(int)
    income = np.exp(rng.normal(11.0, 0.45, n)) * (1 + (age - 19) / 120)
    income = income.round(-2)
    emp_years = np.clip((age - 19) * rng.uniform(0.1, 0.8, n), 0, 40).round(1)
    home = rng.choice(["RENT", "MORTGAGE", "OWN"], n, p=[0.45, 0.40, 0.15])
    purpose = rng.choice(["debt_consolidation", "home_improvement", "car", "medical", "small_business", "holiday"],
                         n, p=[0.35, 0.20, 0.17, 0.10, 0.08, 0.10])
    amount = np.exp(rng.normal(9.4, 0.6, n)).round(-2)
    util = np.clip(rng.beta(2.2, 3.0, n), 0, 1).round(2)
    late = rng.poisson(0.4 + 2.2 * util)
    region = rng.choice(["North", "South", "East", "West"], n, p=[0.28, 0.30, 0.22, 0.20])
    dates = pd.to_datetime("2023-01-01") + pd.to_timedelta(rng.integers(0, 730, n), unit="D")

    # hidden risk -> default
    z = (-3.1 + 2.4 * (util - 0.4) + 0.35 * late
         - 0.9 * (np.log(income) - 11.0) + 0.45 * (purpose == "small_business")
         + 0.25 * (home == "RENT") + 0.35 * (amount / income > 0.35)
         + rng.normal(0, 0.5, n))
    p = 1 / (1 + np.exp(-z))
    default = (rng.uniform(size=n) < p).astype(int)

    # interest rate is a SYMPTOM of risk (risk-based pricing), not a cause
    rate = np.clip(6 + 9 * p + rng.normal(0, 2.2, n), 3.5, 29).round(2)

    df = pd.DataFrame({
        "loan_id": np.arange(10001, 10001 + n), "application_date": dates, "age": age,
        "annual_income": income, "employment_years": emp_years, "home_ownership": home,
        "loan_purpose": purpose, "loan_amount": amount, "interest_rate": rate,
        "credit_utilisation": util, "late_payments_12m": late, "region": region,
        "defaulted": default})

    # ---- plant the clues ----
    # 1) Missing income: MORE likely if defaulted (missing is NOT random!)
    miss_p = np.where(df.defaulted == 1, 0.22, 0.05)
    df.loc[rng.uniform(size=n) < miss_p, "annual_income"] = np.nan
    df.loc[rng.uniform(size=n) < 0.03, "employment_years"] = np.nan
    # 2) Impossible ages
    bad = rng.choice(n, 6, replace=False)
    df.loc[bad, "age"] = [0, 0, 150, 999, 5, 130]
    # 3) Messy labels
    m = rng.uniform(size=n)
    df.loc[(df.home_ownership == "RENT") & (m < 0.15), "home_ownership"] = "rent"
    df.loc[(df.home_ownership == "RENT") & (m > 0.9), "home_ownership"] = "Rent "
    df.loc[(df.home_ownership == "OWN") & (m < 0.3), "home_ownership"] = "own"
    # 4) A typo'd monster loan + a negative income
    df.loc[df.sample(1, random_state=1).index, "loan_amount"] = 9_500_000
    df.loc[df.sample(1, random_state=2).index, "annual_income"] = -45_000
    # 5) Duplicate rows
    df = pd.concat([df, df.sample(25, random_state=3)], ignore_index=True)
    return df.sample(frac=1, random_state=11).reset_index(drop=True)

df_raw = make_case_file()
print('Case file loaded ✅')

**Golden rule #1 of detective work: never touch the original evidence.**
We keep `df_raw` untouched and do all our work on a copy called `df`. If you ever mess up, re-run this cell and you're back to square one.

In [ ]:
df = df_raw.copy()

### 📖 Data dictionary (your case notes)

| Column | Meaning |
|---|---|
| `loan_id` | Unique ID of the loan |
| `application_date` | When the customer applied |
| `age` | Applicant age (years) |
| `annual_income` | Declared yearly income |
| `employment_years` | Years in current employment |
| `home_ownership` | RENT / MORTGAGE / OWN |
| `loan_purpose` | What the loan is for |
| `loan_amount` | Amount borrowed |
| `interest_rate` | Rate charged (%) |
| `credit_utilisation` | Share of existing credit limits being used (0 to 1) |
| `late_payments_12m` | Number of late payments in the last 12 months |
| `region` | Customer region |
| **`defaulted`** | **1 = customer defaulted, 0 = repaid. This is our "crime".** |

## 🔎 Chapter 1: Arrive at the scene

A detective doesn't start with theories. They start by *looking*: How big is this place? What's lying around? Does anything look off?

**Core pandas idea:** a `DataFrame` is a table. Rows = records (here, loans). Columns = variables. A single column is a `Series`.

> 🔮 **Prediction:** How many rows and columns do you think `df` has? Roughly what share of loans defaulted? Write your guesses, then run.

In [ ]:
df.shape   # (rows, columns)

In [ ]:
df.head()   # the first 5 rows. Try df.head(10), or df.tail(), or df.sample(5)

In [ ]:
df.info()   # column names, data types, and non-null counts

👀 **Look closely at `info()`:** the `Non-Null Count` column is a free missing-data report. If a column has fewer non-nulls than the total rows, something is missing. Which columns?

Now the numeric summary:

In [ ]:
df.describe()   # count, mean, std, min, quartiles, max for numeric columns

### 🕵️ Detective's checklist for `describe()`
Read it like a suspicious investigator, not a student:
- **`min` / `max`:** any impossible values? (An age of 0? Negative income?)
- **`mean` vs `50%` (median):** if the mean is far above the median, the column is *skewed* or has extreme outliers.
- **`count`:** does it match the number of rows?

> 🗣️ **Explain it:** In 2 sentences, write down every suspicious thing you spotted in the table above. (There are at least 4. Hunt for them!)

In [ ]:
# 🎯 YOUR TURN: what share of ALL loans defaulted?
# Hint: 'defaulted' is 0/1, so the mean of the column is the share of 1s.
# Write your code here

In [ ]:
# 🔓 SOLUTION (try first!)
df['defaulted'].mean()

💡 **Insight:** the mean of a 0/1 column *is* a rate. You'll use this trick constantly: **default rate = `mean()` of the default flag.**
This is called the **base rate**, and everything else we find should be compared to it. A group with a 10% default rate is unremarkable. A group with 30% is a clue.

## 🕳️ Chapter 2: What's missing from the evidence?

Missing data is never just an annoyance. **The pattern of what's missing is often a clue in itself.**
In pandas, a missing value is `NaN` ("not a number"). The tools: `isna()` gives True/False per cell, and `sum()` / `mean()` count them, because True = 1 and False = 0.

In [ ]:
df.isna().sum()   # how many missing per column?

In [ ]:
(df.isna().mean() * 100).round(1)   # what PERCENT is missing per column?

> 🔮 **Prediction:** `annual_income` is missing for some loans. Do you think those customers **default more, less, or about the same** as everyone else? Commit to a guess, *then* run the next cell.

In [ ]:
# Split loans into two groups: income missing vs. income present, then compare default rates.
df.groupby(df['annual_income'].isna())['defaulted'].mean()

### 🚨 Plot twist
Customers with **missing income default far more often.** That means the missingness is **not random**. Perhaps customers who were already in trouble avoided stating their income, or a particular sales channel skipped the field.

**Big lesson:** if you had lazily dropped rows with missing income, you'd have thrown away the *riskiest* customers and made your portfolio look safer than it is. Never delete missing data until you've asked: *"Why is it missing?"*

Let's turn "is it missing?" into a proper column so we can use it later (a classic trick called a *missing indicator*):

In [ ]:
df['income_missing'] = df['annual_income'].isna()
df['income_missing'].value_counts()

In [ ]:
# 🎯 YOUR TURN: Is employment_years also missing in a suspicious way?
# Compare the default rate for rows where employment_years is missing vs present.
# Write your code here

In [ ]:
# 🔓 SOLUTION (try first!)
df.groupby(df['employment_years'].isna())['defaulted'].mean()

🗣️ **Explain it:** Why can *missing values* be informative? Compare what you saw for `annual_income` and `employment_years`. (Hint: check the size of the groups, since a small group gives a shakier rate.)

⚡ **Recall (answer tomorrow, no peeking):** What does `df.isna().mean()` return? Why does `.mean()` work on True/False values?

## 🚩 Chapter 3: Suspicious witnesses (errors, outliers, duplicates)

Some data is wrong. Not "interesting"-wrong. *Impossible*-wrong. A detective spots the witness whose story can't be true.

### Clue A: Impossible ages
Back in `describe()` the age column had weird extremes. Let's catch the culprits using **filtering**. `df[condition]` keeps only rows where the condition is True.

In [ ]:
df[(df['age'] < 18) | (df['age'] > 100)]   # | means OR, & means AND. Parentheses are required!

Same filter, more readable, using `query()`:

In [ ]:
df.query('age < 18 or age > 100')[['loan_id', 'age', 'annual_income', 'defaulted']]

These are data-entry errors (nobody is 999, and nobody under 18 can get a loan). Let's look at the **other** extreme suspects: the biggest loans and the weirdest incomes. `sort_values()` puts the extremes on top.

In [ ]:
df.sort_values('loan_amount', ascending=False).head(5)[['loan_id', 'loan_amount', 'annual_income', 'defaulted']]

In [ ]:
df.sort_values('annual_income').head(5)[['loan_id', 'annual_income', 'loan_amount']]

> 🗣️ **Explain it:** One loan is for about 9.5 million and one customer has a *negative* income. For each, say whether you think it's a **real extreme** or a **data error**, and *how you could find out* in a real job. (Real analysts ask the data owner. Pandas can't tell you the truth, only point at the suspect.)

### Clue B: Duplicate rows
The same loan recorded twice would double-count outcomes.

In [ ]:
df.duplicated().sum()   # how many rows are exact copies of an earlier row?

In [ ]:
df[df.duplicated(keep=False)].sort_values('loan_id').head(6)   # keep=False shows BOTH copies side by side

### Clue C: Messy labels
Computers are literal: `'RENT'`, `'rent'` and `'Rent '` (with a trailing space!) are three *different* values.
> 🔮 **Prediction:** How many distinct values will `home_ownership` show? You'd hope for 3.

In [ ]:
df['home_ownership'].value_counts()

Six categories instead of three. If we'd run a `groupby` now, the renters would be split across three groups and every analysis would silently be wrong. **Silent errors are the most dangerous kind.**

⚡ **Recall (tomorrow):** Name the three kinds of "bad evidence" in this chapter and the pandas function you used to find each.

## 🧼 Chapter 4: Clean the evidence (without destroying it)

Rules of a good clean-up:
1. Work on the **copy** (`df`), never `df_raw`.
2. **Write down every decision** and why you made it. (Future you, and your boss, will ask.)
3. Prefer *flagging* over *deleting* when unsure.
4. Print row counts before and after, so you always know what you changed.

In [ ]:
print('Rows before:', len(df))

# 1) Remove exact duplicate rows
df = df.drop_duplicates()

# 2) Standardise text labels: strip spaces, then make lower-case
df['home_ownership'] = df['home_ownership'].str.strip().str.lower()

# 3) Impossible ages -> missing (we can't know the true age, so we say 'unknown' instead of inventing one)
df.loc[(df['age'] < 18) | (df['age'] > 100), 'age'] = np.nan

# 4) Negative income is impossible -> missing
df.loc[df['annual_income'] < 0, 'annual_income'] = np.nan

# 5) Flag (don't delete!) the monster loan so we can decide later
df['loan_outlier'] = df['loan_amount'] > 1_000_000

print('Rows after: ', len(df))

In [ ]:
df['home_ownership'].value_counts()   # much better: 3 categories

In [ ]:
df[['age', 'annual_income']].describe().loc[['count', 'min', 'max']]

✅ Notice we **didn't fill in** missing income with an average. That's a decision you can make later (it's called *imputation*) but it's easy to do badly, and here missingness itself is predictive, so we kept the `income_missing` flag.

> 🎯 **Your turn:** the biggest loan is a 9.5M outlier. Create `df_clean` that excludes loans flagged as outliers, and confirm it has exactly one fewer row.

In [ ]:
# Write your code here

In [ ]:
# 🔓 SOLUTION (try first!)
df_clean = df[~df['loan_outlier']].copy()   # ~ means NOT
print(len(df), '->', len(df_clean))

From here on we use `df_clean`.

In [ ]:
df = df_clean

## 🗣️ Chapter 5: Interview one witness at a time (univariate analysis)

Before comparing variables, understand each one alone. For **categories** → count them. For **numbers** → look at their *shape*.

### Categories: `value_counts()`

In [ ]:
df['loan_purpose'].value_counts()

In [ ]:
df['loan_purpose'].value_counts(normalize=True).round(3)   # normalize=True gives proportions instead of counts

### Numbers: the histogram shows the *shape*

In [ ]:
df['credit_utilisation'].plot(kind='hist', bins=30, edgecolor='white', title='Credit utilisation')
plt.xlabel('Utilisation (0 = none used, 1 = maxed out)')
plt.show()

In [ ]:
df['loan_amount'].plot(kind='hist', bins=40, edgecolor='white', title='Loan amount')
plt.show()

> 🔮 **Prediction:** Loan amounts have a long tail to the right (most loans small, a few huge). Do you expect the **mean** to be *higher or lower* than the **median**? Check it below.

In [ ]:
print('mean  :', round(df['loan_amount'].mean()))
print('median:', round(df['loan_amount'].median()))

💡 **Rule of thumb:** with right-skewed data (money almost always is), the **median** describes a "typical" value better than the mean. Banks look at medians for incomes and balances for exactly this reason.

> 🎯 **Your turn:** Plot a histogram of `age`, then of `annual_income`. Which one is more skewed? Which would you summarise with the median?

In [ ]:
# Write your code here

In [ ]:
# 🔓 SOLUTION (try first!)
df['age'].plot(kind='hist', bins=30, edgecolor='white', title='Age'); plt.show()
df['annual_income'].plot(kind='hist', bins=40, edgecolor='white', title='Annual income'); plt.show()

⚡ **Recall (tomorrow):** What's the difference between `value_counts()` and `value_counts(normalize=True)`? When is the median better than the mean?

## 🔗 Chapter 6: Follow the money (`groupby` is your magnifying glass)

This is the single most important pandas skill for EDA. The pattern is always:

> **split** the data into groups → **apply** a calculation → **combine** the results

Read it like a sentence: *"Group by loan purpose, then take the mean of `defaulted`."*

In [ ]:
df.groupby('loan_purpose')['defaulted'].mean().sort_values(ascending=False)

That's a default rate per purpose. But rates alone can mislead: a 40% default rate on 5 loans means nothing. **Always show the group size too.** `agg()` lets us compute several things at once:

In [ ]:
summary = (df.groupby('loan_purpose')
             .agg(n_loans=('defaulted', 'size'),
                  default_rate=('defaulted', 'mean'),
                  avg_amount=('loan_amount', 'mean'))
             .sort_values('default_rate', ascending=False))
summary

In [ ]:
(summary['default_rate'] * 100).plot(kind='barh', title='Default rate by loan purpose (%)')
plt.axvline(df['defaulted'].mean() * 100, color='red', linestyle='--', label='overall rate')
plt.legend(); plt.show()

> 🗣️ **Explain it:** Which purpose stands out? Is the gap big enough to care about? (Look at the red base-rate line **and** `n_loans`.)

> 🎯 **Your turn:** Do the same for `home_ownership` and `region`. Does either look like a real clue, or are they all near the base rate?

In [ ]:
# Write your code here

In [ ]:
# 🔓 SOLUTION (try first!)
for col in ['home_ownership', 'region']:
    print(df.groupby(col).agg(n_loans=('defaulted','size'), default_rate=('defaulted','mean')).round(3), '\n')

### Turning numbers into groups: `pd.cut`
`groupby` works on categories. To use it on a *number* like utilisation, first chop the number into bands (**binning**).

In [ ]:
df['util_band'] = pd.cut(df['credit_utilisation'],
                         bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
                         labels=['0-20%', '20-40%', '40-60%', '60-80%', '80-100%'],
                         include_lowest=True)

util_summary = df.groupby('util_band', observed=True).agg(n_loans=('defaulted', 'size'), default_rate=('defaulted', 'mean'))
util_summary

In [ ]:
(util_summary['default_rate'] * 100).plot(kind='bar', rot=0, title='Default rate by credit utilisation band (%)')
plt.ylabel('% defaulted'); plt.show()

🎉 **That's your first real clue.** Default risk climbs steadily as utilisation rises. Notice the *shape*: it isn't a coin-flip difference, it's a clear **gradient**. Gradients (monotonic trends) are what risk models love.

> 🎯 **Your turn:** Do the same for `late_payments_12m`. Group by it directly (no binning needed since it's already a small whole number). What do you see?

In [ ]:
# Write your code here

In [ ]:
# 🔓 SOLUTION (try first!)
df.groupby('late_payments_12m').agg(n_loans=('defaulted','size'), default_rate=('defaulted','mean')).round(3)

⚡ **Recall (tomorrow):** Explain **split → apply → combine** in your own words. Why must you always show `n_loans` next to a rate?

## 🧩 Chapter 7: Connect the dots (two variables at once)

### Correlation: a quick radar
`corr()` gives a number between -1 and +1 for each pair of numeric columns. Positive = rise together. Negative = one rises as the other falls. Near 0 = no *straight-line* relationship.

In [ ]:
num_cols = ['age', 'annual_income', 'employment_years', 'loan_amount', 'interest_rate',
            'credit_utilisation', 'late_payments_12m', 'defaulted']
df[num_cols].corr()['defaulted'].drop('defaulted').sort_values(ascending=False).round(3)

> ⚠️ **Detective's caution:** correlation is a *radar*, not a verdict. It only sees straight-line relationships, is shaken by outliers, and says nothing about cause.

A more visual way to compare a number across outcomes, the **boxplot**:

In [ ]:
df.boxplot(column='credit_utilisation', by='defaulted', grid=False)
plt.suptitle(''); plt.title('Credit utilisation: repaid (0) vs defaulted (1)'); plt.show()

### Crosstab: two categories at once
`pd.crosstab` counts combinations. With `normalize='index'` each row sums to 100%, giving a default rate per row.

In [ ]:
pd.crosstab(df['home_ownership'], df['defaulted'], normalize='index').round(3)

### 🔥 Combining two clues
Real risk usually comes from *combinations*. Let's look at utilisation band **and** late payments together, using a pivot table of default rates:

In [ ]:
df['late_flag'] = np.where(df['late_payments_12m'] >= 2, '2+ late', '0-1 late')
pd.pivot_table(df, values='defaulted', index='util_band', columns='late_flag', aggfunc='mean', observed=True).round(3)

> 🗣️ **Explain it:** Which corner of this table is the most dangerous? Which is the safest? How many times riskier is the worst corner than the best? (Divide the two numbers.)

> 🎯 **Your turn:** Make the same pivot using `home_ownership` as columns instead. Does the table show a strong pattern, or does it look flat?

In [ ]:
# Write your code here

In [ ]:
# 🔓 SOLUTION (try first!)
pd.pivot_table(df, values='defaulted', index='util_band', columns='home_ownership', aggfunc='mean', observed=True).round(3)

## 🐟 Chapter 8: Beware the red herring (correlation ≠ causation)

You'll notice `interest_rate` is positively correlated with default. Tempting headline: **"High interest rates cause defaults! Cut the rates!"**

In [ ]:
df.groupby(pd.cut(df['interest_rate'], [0, 8, 10, 12, 14, 30]), observed=True).agg(
    n_loans=('defaulted', 'size'), default_rate=('defaulted', 'mean')).round(3)

> 🔮 **Prediction (important!):** Think about how a bank *sets* interest rates. Would a bank charge a **safe** customer or a **risky** customer a higher rate? So which way does the arrow of cause likely point?

**The deduction:** banks use **risk-based pricing**. They already *suspected* these borrowers were risky (high utilisation, late payments) and charged them more. The rate is a **symptom** of risk the bank already saw, not necessarily the cause of default. (Cutting rates for everyone wouldn't make risky customers safe.)

This is a classic trap in credit modelling too: variables that are *outputs* of an earlier risk decision can look like great predictors while adding nothing causal, and may not even be available at the time you need to predict.

🗣️ **Explain it:** Give one more example from banking or everyday life where A and B move together but A doesn't cause B.

**Three questions to ask of every pattern** (stick this on your wall):
1. **Is it real?** (Enough rows? Not just noise or an artifact of bad data?)
2. **Is it causal, or is there a hidden third factor?** (Confounding, reverse causality.)
3. **Is it useful?** (Would we know this variable *at decision time*? Can anyone act on it?)

## 📝 Chapter 9: Write the case report

An analysis nobody can understand is worthless. Complete this report **in your own words** (double-click the cell to edit). Aim for plain English a manager would get in 30 seconds.

---
**CASE REPORT: Why do Meridian loans default?**

**1. Data quality issues found & what I did about them**
- …

**2. Overall default rate:** …%

**3. Top 3 risk clues (with numbers):**
- Clue 1: …
- Clue 2: …
- Clue 3: …

**4. Red herrings / things I should NOT conclude:**
- …

**5. What I'd do next (questions for the data owner, extra data I want, a model to try):**
- …
---

## 🏆 Boss level (do this without hints)

Pick **one** and complete it on your own. Struggling here is where the real learning happens.

1. **Time detective:** Convert `application_date` to a month (`df['application_date'].dt.to_period('M')`) and plot the default rate by month. Is risk getting better or worse over time? Be careful: are there enough loans in every month?
2. **Affordability detective:** Create a new column `loan_to_income = loan_amount / annual_income`. Bin it, and check how default rate changes. (Remember what you learned about missing income!)
3. **Young vs old:** Bin `age` into bands (18-25, 26-35, …) and find which band is riskiest. Then check whether the pattern *survives* after splitting by `home_ownership`. (Does a hidden third factor explain it?)

## ⚡ Retrieval practice, final round (tomorrow, closed notebook)

Write answers from memory first, *then* check:
1. What does `df.groupby('x')['y'].mean()` do, in words?
2. How do you count missing values per column? How do you get percentages?
3. Why did we avoid just dropping rows with missing income?
4. What's the difference between `df[...]`, `.query()`, and `.loc[...]` for filtering?
5. Why did we check `n_loans` beside every default rate?
6. Why is `interest_rate` a suspicious "explanation" of default?

## 🧰 Cheat sheet (the EDA toolkit you just earned)

| Question | Code |
|---|---|
| How big is it? | `df.shape` |
| What are the columns & types? | `df.info()` |
| Summary of numbers | `df.describe()` |
| Missing count / % | `df.isna().sum()` / `df.isna().mean()` |
| Duplicates | `df.duplicated().sum()` → `df.drop_duplicates()` |
| Filter rows | `df[df['age'] > 30]` or `df.query('age > 30')` |
| Sort | `df.sort_values('col', ascending=False)` |
| Count categories | `df['col'].value_counts(normalize=True)` |
| Clean text | `df['col'].str.strip().str.lower()` |
| Rate by group | `df.groupby('col')['target'].mean()` |
| Many stats at once | `.agg(n=('target','size'), rate=('target','mean'))` |
| Bin a number | `pd.cut(df['col'], bins=[...])` |
| Two-way table | `pd.crosstab(a, b, normalize='index')` / `pd.pivot_table(...)` |
| Correlations | `df[cols].corr()` |
| Quick plots | `.plot(kind='hist' / 'bar' / 'barh')`, `df.boxplot(...)` |

**The detective's loop:** *Look → Doubt → Hypothesise → Test → Ask "so what?" → Repeat.*